In [53]:
import omidb
import shutil
import os
import pickle
import pydicom
import png
import numpy as np

OMIDB_DATA_PATH = '/Users/hendrik/Studium/Master/Thesis/Data/OMI-DB Sample/DATA'
OMIDB_IMAGES_PATH = '/Users/hendrik/Studium/Master/Thesis/Data/OMI-DB Sample/IMAGES'
db = omidb.DB(OMIDB_DATA_PATH, OMIDB_IMAGES_PATH, ignore_missing_images=False)

In [30]:
clients = [client for client in db]
[client.id for client in clients]

['demd1457',
 'demd82216',
 'demd4811',
 'demd146067',
 'demd286201',
 'demd63958',
 'demd142225',
 'demd5951',
 'demd7632',
 'demd135753',
 'demd101562',
 'demd135085',
 'demd113215',
 'demd30457',
 'demd84763',
 'demd57407',
 'demd5582',
 'demd100952',
 'demd135302',
 'demd62919',
 'demd12017',
 'demd193385',
 'demd240719',
 'demd4137',
 'demd2618',
 'demd3786',
 'demd54409',
 'demd216476',
 'demd152378',
 'demd25280',
 'demd2575',
 'demd6141',
 'demd225344',
 'demd107840',
 'demd72693',
 'demd82034',
 'demd103482',
 'demd73791',
 'demd5496',
 'demd131461',
 'demd54394',
 'demd198481',
 'demd212895',
 'demd151045',
 'demd62902',
 'demd45173',
 'demd7639',
 'demd178158',
 'demd136376',
 'demd3770']

In [24]:
c = 0
for client in clients:
    if c > 2:
        break
    c += 1
    print(f"Client {c}: {client.id}")
    for episode in client.episodes:
        print(f"Episode {episode.id}")
        if episode.studies:
            for study in episode.studies:
                print(f"Study {study.id}")
        else:
            print("No studies in episode")
        


Client 1: demd1457
Episode 9999
No studies in episode
Episode 9994
Study 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0
Study 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2734.0
Client 2: demd82216
Episode 9992
Study 1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.55.0
Episode 9986
No studies in episode
Episode 9999
Study 1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.76.0
Client 3: demd4811
Episode 9969
No studies in episode
Episode 9998
No studies in episode
Episode 9996
No studies in episode
Episode 9997
No studies in episode
Episode 9961
Study 1.2.826.0.1.3680043.9.3218.1.1.17326760.1030.1512001788563.317.0
Episode 9957
No studies in episode


In [7]:
1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0

9999
9994


In [8]:
example_episode = example_client.episodes[1]
example_studies = example_episode.studies
example_study = example_episode.studies[0]
example_series = example_study.series[0]
example_images = example_series.images
example_image = example_series.images[0]
example_dcm = example_image.dcm
example_laterality = example_dcm.ImageLaterality
num_imgs = example_series.num_images
print("Number of studies in first episode:", len(example_studies))
print("Number of series in first study:", len(example_study.series))
print("Number of images in first series:", len(example_series.images))
print("Study ID:", example_study.id)
id1="1.2.826.0.1.3680043.9.3218.1.1.16668021.8024.1539265126911.241.0"
id2="1.2.826.0.1.3680043.9.3218.1.1.16668021.8024.1539265126911.258.0"
print(example_study.id == id1)
print(example_study.id == id2)
# pprint(example_images)
# pprint(example_dcm)
# pprint(example_laterality)
# pprint(example_dcm)
# pprint(example_series.images)

Number of studies in first episode: 2
Number of series in first study: 8
Number of images in first series: 1
Study ID: 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0
False
False


In [31]:
#Utility functions that I will need later

def study_is_screening_event(study):
    if omidb.events.Event.screening in study.event_type:
        return True
    else:
        return False

def get_studies_with_at_least_four_series(episodes):
    studies_with_at_least_four_series = []
    for episode in episodes:
        if episode.studies:
            for study in episode.studies:
                if study.series:
                    if len(study.series) >= 4:
                        studies_with_at_least_four_series.append(study.id)
        else:
            continue
    return studies_with_at_least_four_series

def get_view_from_dcm(dcm):
    laterality = dcm.ImageLaterality
    view_position = dcm.ViewPosition
    full_view = f"{laterality}-{view_position}"
    return full_view

def get_presentation_intent_type_from_dcm(dcm):
    return dcm.PresentationIntentType

def get_image_id_from_dcm(dcm):
    return dcm.SOPInstanceUID

def create_img_path(client_id, study_id, img_id):
    return f"{client_id}/{study_id}/{img_id}.dcm"


In [32]:
relevant_studies = get_studies_with_at_least_four_series(example_client.episodes)
pprint(relevant_studies)

['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0',
 '1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2734.0']


In [ ]:
for i in range(len(example_study.series)):
    img = example_study.series[i].images[0]
    example_study.series[i].plot()
    print(get_presentation_intent_type_from_dcm(img.dcm))
    print(get_view_from_dcm(img.dcm))
    print(get_image_id_from_dcm(img.dcm))

In [36]:
def create_exam_list(clients):
    exam_list = []
    exam_list_img_paths = []
    for client in clients:
        client_id = client.id
        for episode in client.episodes:
            for study in episode.studies:
                study_id = study.id
                study_dict = {
                    'horizontal_flip': 'NO',
                    'L-CC': [],
                    'L-MLO': [],
                    'R-MLO': [],
                    'R-CC': []
                }
                study_img_paths =[]
                if len(study.series) >= 4:
                    for series in study.series:
                        print(f"Number of images in series {series.id}: {series.num_images}")
                        if len(series.images) == 1:
                            img = series.images[0]
                            view = get_view_from_dcm(img.dcm)
                            img_id = get_image_id_from_dcm(img.dcm)
                            if get_presentation_intent_type_from_dcm(img.dcm) == "FOR PRESENTATION":
                                study_dict[view] = [img_id]
                                img_path = create_img_path(client_id, study_id, img_id)
                                study_img_paths.append(img_path)
                        else:
                            print(f"Series {series.id} contains {len(series.images)} images. Skipping.")
                            continue
                # Now check if all views have images (non-empty lists)
                if (study_dict['L-CC'] and study_dict['L-MLO'] and 
                    study_dict['R-MLO'] and study_dict['R-CC']):
                    exam_list.append(study_dict)
                    exam_list_img_paths.append(study_img_paths)
                else:
                    print(f"Study {study.id} does not contain all views. Skipping.")
                    continue
    return exam_list, exam_list_img_paths

In [37]:
exam_list, exam_list_img_paths = create_exam_list(clients)

Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2722.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2720.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2726.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2714.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2732.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2716.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2728.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2710.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2738.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2740.0: 1
Number of images in series 1.2.826.0.1.3680043.9.3

In [16]:
pprint(exam_list)


[{'L-CC': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2718.0'],
  'L-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2730.0'],
  'R-CC': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2712.0'],
  'R-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2724.0'],
  'horizontal_flip': 'NO'},
 {'L-CC': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.65.0'],
  'L-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.71.0'],
  'R-CC': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.61.0'],
  'R-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.74.0'],
  'horizontal_flip': 'NO'},
 {'L-CC': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.88.0'],
  'L-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.92.0'],
  'R-CC': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.98.0'],
  'R-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.

In [38]:
print(exam_list_img_paths)

[['demd1457/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2724.0.dcm', 'demd1457/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2718.0.dcm', 'demd1457/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2730.0.dcm', 'demd1457/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2709.0/1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2712.0.dcm'], ['demd82216/1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.55.0/1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.65.0.dcm', 'demd82216/1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.55.0/1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.71.0.dcm', 'demd82216/1.2.826.0.1.3680043.9.3218.1.1.244892981.2499.1540047254554.55.0/1.2.826.0.1.3680043.9.3218.1.1.244892981.2

In [41]:
print(exam_list[0])

{'horizontal_flip': 'NO', 'L-CC': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2718.0'], 'L-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2730.0'], 'R-MLO': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2724.0'], 'R-CC': ['1.2.826.0.1.3680043.9.3218.1.1.3850610.1631.1511384163736.2712.0']}


In [46]:
# Store the first NUM_EXAMS exams from exam_list and store exam_labels as a pickle file
exam_list_output_path = "/Users/hendrik/Studium/Master/Thesis/Code/breast_cancer_classifier/omi_db_data/"
num_exams = 2
exam_list = exam_list[:num_exams]
# Store exam list as pickle file
try:
    with open(exam_list_output_path + "exam_list_before_cropping.pkl", 'wb') as f:
        pickle.dump(exam_list, f)
except Exception as e:
    print(f"Error while saving pickle file: {e}")


In [47]:
def save_dicom_image_as_png(dicom_filename, png_filename, bitdepth=12):
    """
    Save 12-bit mammogram from dicom as rescaled 16-bit png file.
    :param dicom_filename: path to input dicom file.
    :param png_filename: path to output png file.
    :param bitdepth: bit depth of the input image. Set it to 12 for 12-bit mammograms.
    """
    try:
        dicom = pydicom.read_file(dicom_filename)
        image = dicom.pixel_array

        # Normalize the pixel values to fit into the specified bit depth
        max_pixel_value = np.max(image)
        scale_factor = (2 ** bitdepth - 1) / max_pixel_value
        image = (image * scale_factor).astype(np.uint16)  # Ensure image is of type uint16

        with open(png_filename, 'wb') as f:
            writer = png.Writer(height=image.shape[0], width=image.shape[1], bitdepth=bitdepth, greyscale=True)
            writer.write(f, image.tolist())
    except FileNotFoundError:
        print(f"File {dicom_filename} not found.")
    except Exception as e:
        print(f"Error while saving {dicom_filename} as PNG: {e}")

In [56]:
# Store the first NUM_EXAMS exams as png images in img_output_path
img_output_path = "/Users/hendrik/Studium/Master/Thesis/Code/breast_cancer_classifier/omi_db_data/images/"
img_source_path = "/Users/hendrik/Studium/Master/Thesis/Data/OMI-DB Sample/IMAGES/"

for i in range(num_exams):
    for j in range(len(exam_list_img_paths[i])):
        img_path = exam_list_img_paths[i][j]
        # Get only the filename without the extension
        img_filename_with_extension = img_path.split("/")[-1]
        # Remove the extension by splitting only on the last dot
        img_filename = img_filename_with_extension.rsplit(".", 1)[0]
        save_dicom_image_as_png(img_source_path + img_path, img_output_path + img_filename + ".png")
